<a href="https://colab.research.google.com/github/Limsungrae/Hongong/blob/main/%ED%98%BC%EA%B3%B5%EB%A8%B8%EC%8B%A05_2%EA%B5%90%EC%B0%A8%EA%B2%80%EC%A6%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 검증 세트

In [2]:
import pandas as pd
wine = pd.read_csv('http://bit.ly/wine_csv_data')
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [3]:
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

In [4]:
from numpy.random import rand
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

In [5]:
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target,test_size=0.2,random_state=42)

In [6]:
print(sub_input.shape, val_input.shape)

(4157, 3) (1040, 3)


In [7]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)

print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


## 교차 검증

In [8]:
from sklearn.model_selection import cross_validate

scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.07155919, 0.04437995, 0.03965259, 0.01808047, 0.01360154]), 'score_time': array([0.01195908, 0.00797081, 0.00567341, 0.00306177, 0.00751972]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [10]:
import numpy as np

print(np.mean(scores['test_score']))

0.855300214703487


In [11]:
from sklearn.model_selection import StratifiedKFold

scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [12]:
# StratifiedKFold로 교차검증 수행
# 평균 정확도 출력

splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score']))

0.8574181117533719


## 하이퍼파라미터 튜닝 (GridSearch)

In [13]:

from sklearn.model_selection import GridSearchCV
# 탐색할 파라미터 설정 (불순도 감소 기준)
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [14]:
# GridSearchCV 객체 생성 (모든 조합 탐색)

gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

In [15]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [17]:
# 전체 훈련 데이터 정확도 출력
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [19]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [20]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [22]:
print(gs.cv_results_['params'][gs.best_index_])

{'min_impurity_decrease': 0.0001}


In [23]:
# 더 많은 파라미터 조합 설정
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10)
          }

In [24]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': range(5, 20),
                         'min_impurity_decrease': array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009]),
                         'min_samples_split': range(2, 100, 10)})

In [25]:
print(gs.best_params_)

{'max_depth': 14, 'min_impurity_decrease': np.float64(0.0004), 'min_samples_split': 12}


In [26]:
# 최고 성능 출력
print(np.max(gs.cv_results_['mean_test_score']))

0.8683865773302731


## 랜덤 서치

In [29]:
from scipy.stats import uniform, randint  # 확률 분포 생성

# 정수 랜덤 생성 (0~9)
rgen = randint(0, 10)
rgen.rvs(10)  # 10개 샘플 생성

# 분포 확인
np.unique(rgen.rvs(1000), return_counts=True)

# 실수 랜덤 생성 (0~1)
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.78643048, 0.55515823, 0.55063687, 0.87496844, 0.83846711,
       0.2738206 , 0.46230493, 0.86473456, 0.77101901, 0.67168106])

In [30]:
# 랜덤 탐색용 파라미터 범위 설정
params = {
    'min_impurity_decrease': uniform(0.0001, 0.001),  # 연속값
    'max_depth': randint(20, 50),
    'min_samples_split': randint(2, 25),
    'min_samples_leaf': randint(1, 25),
}

In [31]:
from sklearn.model_selection import RandomizedSearchCV  # 랜덤 서치

# 100번 랜덤 샘플링하여 탐색
rs = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42),
    params,
    n_iter=100,  # 시도 횟수
    n_jobs=-1,
    random_state=42
)

# 학습
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x78f2d6272ff0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x78f2d6271910>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x78f2d6273830>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x78f2d6272750>},
                   random_state=42)

In [32]:
# 최적 파라미터 출력
print(rs.best_params_)

# 최고 성능 출력
print(np.max(rs.cv_results_['mean_test_score']))

# 최적 모델
dt = rs.best_estimator_

# 테스트 데이터 정확도 (최종 성능)
print(dt.score(test_input, test_target))

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}
0.8695428296438884
0.86


In [33]:
# splitter='random' 옵션 추가 (랜덤하게 분할)
gs = RandomizedSearchCV(
    DecisionTreeClassifier(splitter='random', random_state=42),
    params,
    n_iter=100,
    n_jobs=-1,
    random_state=42
)

# 학습
gs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42,
                                                    splitter='random'),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x78f2d6272ff0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x78f2d6271910>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x78f2d6273830>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x78f2d6272750>},
                   random_state=42)

In [34]:
# 최적 파라미터
print(gs.best_params_)

# 최고 성능
print(np.max(gs.cv_results_['mean_test_score']))

# 최적 모델
dt = gs.best_estimator_

# 테스트 정확도
print(dt.score(test_input, test_target))

{'max_depth': 43, 'min_impurity_decrease': np.float64(0.00011407982271508446), 'min_samples_leaf': 19, 'min_samples_split': 18}
0.8458726956392981
0.786923076923077
